In [23]:
from unicodedata import decomposition

from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence


In [24]:
loader = TextLoader("Langchain_crewai.txt")
docs = loader.load()

In [36]:
splitter = RecursiveCharacterTextSplitter(chunk_size= 300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)

retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k":5, "lambda_mult":0.7})
chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4085.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(metadata={'source': 'Langchain_crewai.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'Langchain_crewai.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'Langchain_crewai.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard'),
 Document(meta

In [37]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [38]:
llm = init_chat_model('groq:openai/gpt-oss-120b', temperature=0.7)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11e4660d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11e467e50>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [39]:
decomposition_prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Your task is to decompose the user's query into smaller sub-queries that can help retrieve more relevant documents.

    Original Query: "{question}"

    Sub-Queries: """
)

In [40]:
decomposition_chain = decomposition_prompt|llm|StrOutputParser()

In [41]:
query = "How does Langchain use memory and agents compared to CrewAI"

decomposition_question = decomposition_chain.invoke({"question": query})

In [42]:
print(decomposition_question)

**Sub‑Queries**

1. **LangChain Memory Overview**  
   – What types of memory modules does LangChain provide (e.g., conversation, summary, vector, buffer, retriever‑augmented, etc.)?  
   – How are these memory components initialized, stored, and accessed in a LangChain workflow?

2. **LangChain Agents Overview**  
   – What built‑in agents does LangChain offer (e.g., Zero‑Shot ReAct, Conversational, Structured Output, Tool‑using agents)?  
   – How does LangChain orchestrate tool execution and plan generation within an agent?

3. **CrewAI Memory Overview**  
   – What memory mechanisms are available in CrewAI (e.g., shared crew memory, individual role memory, context windows, external knowledge bases)?  
   – How does CrewAI persist and retrieve memory across multiple agents or tasks?

4. **CrewAI Agents Overview**  
   – What are the core agent concepts in CrewAI (e.g., crew members, roles, task delegation, tool‑calling)?  
   – How does CrewAI coordinate multiple agents to accomplis

In [43]:
qa_prompt = PromptTemplate.from_template(
    """
    Use the context below to answer the question. If you don't know the answer, say you don't know.

    Context:{context}

    question: {input}"""
)

qa_chain = create_stuff_documents_chain(llm = llm, prompt = qa_prompt)

In [46]:
def full_query_rag_pipeline(user_query):
    sub_qs = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-1234567890. ").strip() for q in sub_qs.split("\n") if q.strip()]

    results = []

    for sub_q in sub_questions:
        docs = retriever.invoke(sub_q)
        result = qa_chain.invoke({"input": sub_q, "context": docs})
        results.append(f"{sub_q}: {result}")

        return "\n".join(results)


In [47]:
query = "How does Langchain use memory and agents compared to CrewAI"

final_answer = full_query_rag_pipeline(query)
print(final_answer)

**Original Query:** “How does Langchain use memory and agents compared to CrewAI”: **Answer**

Both LangChain and CrewAI can work with memory, but they use it in different ways and for different purposes.

| Aspect | LangChain | CrewAI |
|--------|-----------|--------|
| **Core abstraction** | **Agents** (planner‑executor) that decide which tool to call next. | **Crews** (collections of role‑based agents) that collaborate on a task. |
| **How memory is used** | • Each LangChain agent can be equipped with a *memory* component (e.g., conversation memory, vector‑store retriever, summary buffer). <br>• The memory is consulted **at every step** of the planner‑executor loop so the agent can keep context across a chain of tool calls, perform dynamic branching, and refine its plan based on what happened earlier. | • CrewAI does **not provide its own low‑level memory store** for individual steps. Instead, it can *leverage* the memory that a LangChain agent already has when the crew is built on 